# Assignment 2 — Supervised Workflow

## Objectives and assessment

Build a leakage-free pipeline, score it honestly, separate model selection from performance
estimation, choose a decision threshold from error costs, and assemble the preprocessing for a
table with numeric and categorical columns — the steps of seminar 2, each as one small function.

| Task | Topic | Points |
|---|---|---:|
| 1 | A leakage-free pipeline | 1 |
| 2 | An honest cross-validated score | 2 |
| 3 | Nested cross-validation | 3 |
| 4 | The cheapest decision threshold | 4 |
| 5 | Preprocessing for numeric and categorical columns | 3 |
| **Total** | **13 autograded** | **13** |

## Working instructions

1. Run the setup cell before working on the tasks. All input data is generated in this notebook; no downloads or external files are required.
2. Replace `raise NotImplementedError()` in code answer cells. Preserve function names and signatures and do not edit locked setup or test cells.
3. Return the requested result; printing a value does not return it. Use the function parameters rather than hard-coded results or the global example data.
4. Use exactly the cross-validation objects `CV_OUTER`, `CV_INNER` and the grid `C_GRID` from the setup. Other folds, seeds, solvers or iteration limits give different numbers, and the checks compare against a reference computed with these objects.
5. Restart the kernel, run all cells in order and use **Validate**. Visible checks provide feedback; hidden checks also run every function on a **second dataset you cannot see**, so a function that returns a memorised number will not score.

## Data and task dependencies

| Object | Contents |
|---|---|
| `X`, `y` | 400 synthetic observations, 12 numeric features, 15 % positives (tasks 2–3) |
| `CV_OUTER`, `CV_INNER`, `C_GRID` | The validation objects and the parameter grid for tasks 2–3 |
| `Y_TRUE`, `PROBA` | Ten labels and ten predicted probabilities for task 4 |
| `PASSENGERS` | A six-row table with numeric and categorical columns and missing values for task 5 |

Tasks 2 and 3 call `make_pipeline_lr` from task 1; a wrong step name there also fails task 3,
because `clf__C` addresses the step named `clf`. Tasks 4 and 5 are independent of the others.

## Setup — run first, do not modify

In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=400, n_features=12, n_informative=5, n_redundant=2,
                           weights=[0.85, 0.15], random_state=7)

CV_OUTER = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
CV_INNER = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)
C_GRID = [0.01, 0.1, 1.0, 10.0]

Y_TRUE = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])
PROBA  = np.array([0.05, 0.10, 0.20, 0.35, 0.55, 0.62, 0.30, 0.48, 0.75, 0.90])

PASSENGERS = pd.DataFrame({
    "Age":      [22.0, np.nan, 26.0, 35.0, np.nan, 54.0],
    "Fare":     [7.25, 71.28, 7.92, 53.10, 8.05, 51.86],
    "Sex":      ["male", "female", "female", "female", "male", "male"],
    "Embarked": ["S", "C", "S", np.nan, "S", "S"],
    "Name":     ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "McCarthy"],
})
print("setup complete, positives:", y.mean().round(3))

setup complete, positives: 0.158


## Task 1 — a leakage-free pipeline (1 point)

Write `make_pipeline_lr()` returning a `Pipeline` with exactly two steps, named `"scale"`
(a `StandardScaler`) and `"clf"` (a `LogisticRegression` with `max_iter=1000` and
`random_state=0`). Every call must return a **new, unfitted** pipeline.

In [4]:
def make_pipeline_lr():
    return Pipeline([("scale", StandardScaler()),("clf", LogisticRegression(max_iter=1000, random_state=0)) ])

In [5]:
p = make_pipeline_lr()
assert isinstance(p, Pipeline)
assert [n for n, _ in p.steps] == ["scale", "clf"], f"steps must be named 'scale' and 'clf', got {[n for n, _ in p.steps]}"
assert isinstance(p.named_steps["scale"], StandardScaler) and isinstance(p.named_steps["clf"], LogisticRegression)

## Task 2 — an honest cross-validated score (2 points)

Write `honest_auc(X, y)` returning the **mean** ROC AUC of the pipeline from task 1, evaluated
with `cross_val_score` using `cv=CV_OUTER` and `scoring="roc_auc"`. Pass the pipeline as a
whole: that is what makes the scaler refit inside every fold instead of seeing the validation rows.

In [6]:
def honest_auc(X, y):
    pipe = make_pipeline_lr()
    return cross_val_score(pipe, X, y, cv=CV_OUTER, scoring="roc_auc").mean()

In [7]:
v = honest_auc(X, y)
assert 0.0 <= float(v) <= 1.0

## Task 3 — nested cross-validation (3 points)

Write `nested_auc(X, y)` which

1. builds a `GridSearchCV` over the pipeline from task 1, searching `clf__C` over `C_GRID`, with `scoring="roc_auc"` and `cv=CV_INNER`,
2. passes that search object to `cross_val_score` with `cv=CV_OUTER` and `scoring="roc_auc"`,
3. returns the mean of the outer scores.

The outer loop scores the *whole selection procedure* on rows the inner search never saw; that is the number to report instead of `best_score_`.

In [8]:
def nested_auc(X, y):
    pipe = make_pipeline_lr()
    search = GridSearchCV(pipe, param_grid={ "clf__C": C_GRID }, scoring="roc_auc", cv=CV_INNER)
    return cross_val_score(search, X, y, cv=CV_OUTER, scoring="roc_auc").mean()
    

In [9]:
v = nested_auc(X, y)
assert 0.0 <= float(v) <= 1.0

## Task 4 — the cheapest decision threshold (4 points)

Write `best_threshold(y_true, proba, cost_fp, cost_fn)` returning the threshold that minimises
the total cost `cost_fp * FP + cost_fn * FN`, where a case is predicted positive when
`proba >= threshold` (FP = negatives predicted positive, FN = positives predicted negative).

Which thresholds to try: only those that can change a decision — the **distinct values of
`proba`**, plus `0.0` (everyone predicted positive) and one value above every probability, for
example `max(proba) + 1e-9` (nobody predicted positive). If several thresholds tie, return the
**smallest**. Return a plain number.

Check by hand on `Y_TRUE` and `PROBA` with costs 1 and 10: the default 0.5 flags two negatives
(0.55, 0.62 → cost 2) and misses two positives (0.30, 0.48 → cost 20), total 22. Threshold 0.3
flags three negatives (cost 3) and misses none, total 3 — and no threshold is cheaper.

In [10]:
def expected_cost( y_true, proba, threshold, cost_fp, cost_fn ):
    y_true = np.asarray( y_true )
    proba = np.asarray( proba )
    pred = ( proba >= threshold ).astype( int )
    fp = int( ( ( pred == 1 ) & ( y_true == 0 ) ).sum() )
    fn = int( ( ( pred == 0 ) & ( y_true == 1 ) ).sum() )

    return cost_fp * fp + cost_fn * fn

def best_threshold( y_true, proba, cost_fp, cost_fn ):
    proba = np.asarray( proba )
    candidates = sorted(
        set( proba.tolist() )
        | { 0.0, float( np.max( proba ) ) + 1e-9 }
    )
    return min(candidates, key=lambda threshold: (
            expected_cost( y_true, proba, threshold, cost_fp, cost_fn ),
            threshold
        )
    )

In [11]:
th = best_threshold(Y_TRUE, PROBA, 1.0, 10.0)
assert abs(float(th) - 0.3) < 1e-9, f"with a false negative ten times as expensive, the cheapest threshold is 0.3, got {th}"

## Task 5 — preprocessing for numeric and categorical columns (3 points)

Write `make_preprocessor(numeric, nominal)` returning an **unfitted** `ColumnTransformer` with
exactly two transformers, in this order:

- `"num"` — a `Pipeline` of `SimpleImputer(strategy="median")` followed by `StandardScaler()`, applied to the columns listed in `numeric`;
- `"nom"` — a `Pipeline` of `SimpleImputer(strategy="most_frequent")` followed by `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`, applied to the columns listed in `nominal`.

Columns in neither list are dropped (the `ColumnTransformer` default). Every call returns a new object.

Fitted on `PASSENGERS` with `numeric=["Age", "Fare"]` and `nominal=["Sex", "Embarked"]`, the
transformer returns a 6 × 6 array without missing values: two standardised numeric columns, two
one-hot columns for `Sex`, two for `Embarked` (`Name` is dropped). The hidden checks also fit it
on another table and transform a row with a port of embarkation that was not seen during fitting.

In [12]:
def make_preprocessor( numeric, nominal ):
    numeric_pipeline = Pipeline( [
        ( "imputer", SimpleImputer( strategy="median" ) ),
        ( "scaler", StandardScaler() )
    ] )
    nominal_pipeline = Pipeline( [
        ( "imputer", SimpleImputer( strategy="most_frequent" ) ),
        ( "encoder", OneHotEncoder( handle_unknown="ignore", sparse_output=False) )
    ] )
    return ColumnTransformer( [
        ( "num", numeric_pipeline, numeric ),
        ( "nom", nominal_pipeline, nominal )
    ] )

In [13]:
prep = make_preprocessor(["Age", "Fare"], ["Sex", "Embarked"])
assert isinstance(prep, ColumnTransformer)
out = prep.fit_transform(PASSENGERS)
assert out.shape == (6, 6), f"expected a 6 x 6 array, got {out.shape}"
assert not np.isnan(out).any(), "missing values must be imputed"